# A100 拟合面板坐标诊断

沿用 01 已准备的 LLaMA-2-7B 配置，检验随机坐标的误差覆盖和两种角度搜索在同一坐标对上的收益。每个候选都用物化的共享 Q 重新量化九个面板。默认只检查运行选项。在准备好的 A100 上将下方开关设为 `True` 后运行全部单元。协议和投入判断见 `docs/G1_PAIR_DIAGNOSTIC.md`。

In [ ]:
from pathlib import Path
import os
import sys
from uuid import uuid4

RUN_PAIR_DIAGNOSTIC = False
if os.environ.pop("CELLQUANT_RUN_PAIR_DIAGNOSTIC", "") == "1":
    RUN_PAIR_DIAGNOSTIC = True

if RUN_PAIR_DIAGNOSTIC:
    hint = os.environ.get("CELLQUANT_PROJECT_ROOT")
    candidates = ([Path(hint)] if hint else [Path.cwd(), *Path.cwd().parents, Path("/content/CellQuant")])
    PROJECT_ROOT = next((p.resolve() for p in candidates if (p / "src/cellquant/pair_diagnostic.py").is_file()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError("请将本轮诊断更新解压到 01 使用的 checkout，或设置 CELLQUANT_PROJECT_ROOT。")
    prepared = Path(os.environ.get("CELLQUANT_PILOT_CONFIG", "runs/prepared/g1_gpu/config.json"))
    config_path = PROJECT_ROOT / prepared
    if not config_path.is_file():
        raise FileNotFoundError(f"缺少冻结配置 {config_path}。请用 00 恢复或准备 G1 输入。")
    source = str(PROJECT_ROOT / "src")
    sys.path.insert(0, source)
    os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, [source, os.environ.get("PYTHONPATH")]))
else:
    print("诊断待运行。在已准备的 A100 运行时设置 RUN_PAIR_DIAGNOSTIC = True。")

In [ ]:
if RUN_PAIR_DIAGNOSTIC:
    import json
    from zipfile import ZipFile, ZIP_DEFLATED
    from cellquant.pilot import pilot_method_budget
    from cellquant.runtime import read_json, stream_command

    cfg = read_json(config_path)
    work = PROJECT_ROOT / "runs" / "01_pilot" / ("pair_diagnostic_" + uuid4().hex[:12])
    work.mkdir(parents=True)
    index = work / "index.json"
    command = [sys.executable, "-u", "-m", "cellquant.experiment", "--config", str(config_path),
               "--diagnose-pairs", "--result-index", str(index)]
    print("运行日志", work / "diagnostic.log", flush=True)
    process = stream_command(command, work / "diagnostic.log", cwd=PROJECT_ROOT,
                             timeout_s=3600 + 4 * len(cfg["seeds"]) * pilot_method_budget(cfg))
    if not index.is_file():
        raise RuntimeError(f"运行没有生成索引，详见 {process['log_path']}")
    run = Path(read_json(index)["run_dir"])
    result = read_json(run / "result.json")
    package = work / ("pair_diagnostic_" + run.name + ".zip")
    with ZipFile(package, "w", ZIP_DEFLATED) as archive:
        for name in ["result.json", "config.json", "execution_fingerprint.json", "timings.json",
                     "preflight.json", "model_placement.json", "resources.json"]:
            if (run / name).is_file():
                archive.write(run / name, name)
        for path in sorted(run.glob("seed_*/pair_diagnostic.json")):
            archive.write(path, path.relative_to(run))
    print("结果包", package)
    if process["returncode"] or result["status"] != "DIAGNOSTIC_COMPLETE":
        raise RuntimeError(f"{result['status']} {result.get('reason') or result.get('readiness', {}).get('missing')}")
    print(json.dumps(result["paired_fit_summary"], ensure_ascii=False, indent=2))